# Manufacturing Quality & Data Integrity Monitor

**Goal:** Given batch-level manufacturing data, build a system that:
1. Predicts which production batches are at risk of defects
2. Simulates a realistic "raw ingestion" data layer with common factory data problems
3. Automatically detects and scores data integrity issues per batch
4. Exports everything into clean files ready for a Tableau dashboard

**Dataset:** [Predicting Manufacturing Defects Dataset](https://www.kaggle.com/datasets/rabieelkharoua/predicting-manufacturing-defects-dataset) (Kaggle, by Rabie El Kharoua) — 3,240 production batches with process, supplier, and quality metrics.

**How to run this in Colab:**
1. Upload `manufacturing_defect_dataset.csv` to your Colab session (left sidebar → Files → upload)
2. If you upload it to a different path, update the path in the "Load data" cell below
3. Run all cells top to bottom (`Runtime > Run all`)


In [4]:
# Install/import required libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import os

np.random.seed(42)
pd.set_option('display.max_columns', 30)


## Step 1 — Load the raw Kaggle dataset

This is the untouched dataset: one row per production batch, with process metrics like production volume, supplier quality score, defect rate, maintenance hours, etc.


In [5]:
# CHANGE THIS PATH if you uploaded the CSV somewhere else in Colab
DATA_PATH = 'manufacturing_defect_dataset.csv'

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


Shape: (3240, 17)


,ProductionVolume,ProductionCost,SupplierQuality,DeliveryDelay,DefectRate,QualityScore,MaintenanceHours,DowntimePercentage,InventoryTurnover,StockoutRate,WorkerProductivity,SafetyIncidents,EnergyConsumption,EnergyEfficiency,AdditiveProcessTime,AdditiveMaterialCost,DefectStatus
0,202,13175.403783,86.648534,1,3.121492,63.463494,9,0.052343,8.630515,0.081322,85.042379,0,2419.616785,0.468947,5.551639,236.439301,1
1,535,19770.046093,86.310664,4,0.819531,83.697818,20,4.908328,9.296598,0.038486,99.657443,7,3915.566713,0.119485,9.080754,353.957631,1
2,960,19060.820997,82.132472,0,4.514504,90.350550,1,2.464923,5.097486,0.002887,92.819264,2,3392.385362,0.496392,6.562827,396.189402,1
3,370,5647.606037,87.335966,5,0.638524,67.628690,8,4.692476,3.577616,0.055331,96.887013,8,4652.400275,0.183125,8.097496,164.135870,1
4,206,7472.222236,81.989893,3,3.867784,82.728334,9,2.746726,6.851709,0.068047,88.315554,7,1581.630332,0.263507,6.406154,365.708964,1


## Step 2 — Engineer realistic factory context

The raw Kaggle dataset has no batch IDs, dates, suppliers, or stations — just metrics. Real factory data always has this context, so we add it here to make the dataset behave like something a real MDS (Manufacturing Data Systems) team would work with:

- **BatchID** — unique identifier per production batch
- **ProductionDate** — spread across a 6-month production window
- **SupplierName** — assigned so it loosely correlates with the existing `SupplierQuality` score (better suppliers → higher quality scores), which keeps the simulation realistic
- **StationID / Shift / ProductLine** — where and when the batch was produced

This is the kind of "connect the raw metrics to a traceable context" work described in the JD.


In [6]:
n = len(df)
df = df.reset_index(drop=True)
df['BatchID'] = ['BATCH-' + str(i+1).zfill(6) for i in range(n)]

# Spread production dates across 6 months, roughly increasing (like a real production timeline)
start_date = pd.Timestamp('2026-01-01')
day_offsets = np.sort(np.random.randint(0, 181, size=n))
jitter = np.random.randint(-2, 3, size=n)
df['ProductionDate'] = [start_date + pd.Timedelta(days=int(max(0, d + j))) for d, j in zip(day_offsets, jitter)]

# Assign suppliers, loosely tied to the real SupplierQuality column so it's not pure noise
suppliers = ['Alpha Components', 'Nova Materials', 'Pinnacle Supply', 'Summit Fab', 'Orion Parts', 'Meridian Corp']
sq_bins = pd.qcut(df['SupplierQuality'], q=6, labels=False)
df['SupplierName'] = [suppliers[b] for b in sq_bins]

stations = ['Station-A1', 'Station-A2', 'Station-B1', 'Station-B2', 'Station-C1']
df['StationID'] = np.random.choice(stations, size=n)

shifts = ['Morning', 'Evening', 'Night']
df['Shift'] = np.random.choice(shifts, size=n, p=[0.4, 0.35, 0.25])

lines = ['Line-Assembly-1', 'Line-Assembly-2', 'Line-Test-1']
df['ProductLine'] = np.random.choice(lines, size=n)

df.head()


,ProductionVolume,ProductionCost,SupplierQuality,DeliveryDelay,DefectRate,QualityScore,MaintenanceHours,DowntimePercentage,InventoryTurnover,StockoutRate,WorkerProductivity,SafetyIncidents,EnergyConsumption,EnergyEfficiency,AdditiveProcessTime,AdditiveMaterialCost,DefectStatus,BatchID,ProductionDate,SupplierName,StationID,Shift,ProductLine
0,202,13175.403783,86.648534,1,3.121492,63.463494,9,0.052343,8.630515,0.081322,85.042379,0,2419.616785,0.468947,5.551639,236.439301,1,BATCH-000001,2026-01-01,Pinnacle Supply,Station-A1,Evening,Line-Assembly-2
1,535,19770.046093,86.310664,4,0.819531,83.697818,20,4.908328,9.296598,0.038486,99.657443,7,3915.566713,0.119485,9.080754,353.957631,1,BATCH-000002,2026-01-01,Nova Materials,Station-C1,Morning,Line-Assembly-1
2,960,19060.820997,82.132472,0,4.514504,90.350550,1,2.464923,5.097486,0.002887,92.819264,2,3392.385362,0.496392,6.562827,396.189402,1,BATCH-000003,2026-01-03,Alpha Components,Station-B1,Evening,Line-Assembly-1
3,370,5647.606037,87.335966,5,0.638524,67.628690,8,4.692476,3.577616,0.055331,96.887013,8,4652.400275,0.183125,8.097496,164.135870,1,BATCH-000004,2026-01-02,Pinnacle Supply,Station-C1,Evening,Line-Assembly-2
4,206,7472.222236,81.989893,3,3.867784,82.728334,9,2.746726,6.851709,0.068047,88.315554,7,1581.630332,0.263507,6.406154,365.708964,1,BATCH-000005,2026-01-01,Alpha Components,Station-A1,Morning,Line-Assembly-2


In [7]:
clean_cols = ['BatchID', 'ProductionDate', 'SupplierName', 'StationID', 'Shift', 'ProductLine',
              'ProductionVolume', 'ProductionCost', 'SupplierQuality', 'DeliveryDelay', 'DefectRate',
              'QualityScore', 'MaintenanceHours', 'DowntimePercentage', 'InventoryTurnover',
              'StockoutRate', 'WorkerProductivity', 'SafetyIncidents', 'EnergyConsumption',
              'EnergyEfficiency', 'AdditiveProcessTime', 'AdditiveMaterialCost', 'DefectStatus']
df_clean = df[clean_cols].copy()
df_clean.shape


(3240, 23)

## Step 3 — Simulate a realistic "raw ingestion" layer with data integrity problems

In a real factory, data doesn't arrive perfectly clean — sensors drop connections, systems double-report, values fall outside physical limits, and timestamps drift. We take a copy of the clean data and deliberately inject five common categories of problems:

| Issue type | Real-world cause |
|---|---|
| Duplicate Record | A station resends data after a network hiccup, creating a conflicting duplicate |
| Missing Value | Sensor or connectivity dropout — the field never arrives |
| Out-of-Range Value | Faulty sensor or unit-conversion bug (e.g. negative delay, >100% defect rate) |
| Timestamp Anomaly | Clock drift or misconfigured device reporting a date outside the valid window |
| Orphan Record | Batch missing its station link — can't be traced to a physical location |

This is the "before" state a data integrity system has to catch.


In [8]:
df_raw = df_clean.copy()
issue_log = []
rng = np.random.default_rng(42)

# --- Issue 1: Duplicate records ---
dup_idx = rng.choice(df_raw.index, size=25, replace=False)
dup_rows = df_raw.loc[dup_idx].copy()
dup_rows['DefectRate'] = dup_rows['DefectRate'] * rng.uniform(0.8, 1.2, size=len(dup_rows))
df_raw = pd.concat([df_raw, dup_rows], ignore_index=True)
for bid in dup_rows['BatchID']:
    issue_log.append({'BatchID': bid, 'IssueType': 'Duplicate Record', 'Severity': 'High',
                       'Detail': 'BatchID appears more than once with conflicting values'})

# --- Issue 2: Missing critical fields ---
miss_idx = rng.choice(df_raw.index, size=60, replace=False)
for i in miss_idx:
    field = rng.choice(['SupplierQuality', 'DeliveryDelay', 'MaintenanceHours'])
    df_raw.loc[i, field] = np.nan
    issue_log.append({'BatchID': df_raw.loc[i, 'BatchID'], 'IssueType': 'Missing Value', 'Severity': 'Medium',
                       'Detail': f'{field} missing - likely sensor/connectivity dropout'})

# --- Issue 3: Out-of-range / physically impossible values ---
oor_idx = rng.choice(df_raw.index, size=30, replace=False)
for i in oor_idx:
    choice = rng.integers(0, 3)
    if choice == 0:
        df_raw.loc[i, 'DeliveryDelay'] = -rng.integers(1, 5)
        detail = 'Negative DeliveryDelay (physically impossible)'
    elif choice == 1:
        df_raw.loc[i, 'DefectRate'] = rng.uniform(101, 150)
        detail = 'DefectRate exceeds 100% (out of valid range)'
    else:
        df_raw.loc[i, 'SupplierQuality'] = rng.uniform(101, 130)
        detail = 'SupplierQuality exceeds valid 0-100 range'
    issue_log.append({'BatchID': df_raw.loc[i, 'BatchID'], 'IssueType': 'Out-of-Range Value',
                       'Severity': 'High', 'Detail': detail})

# --- Issue 4: Timestamp anomalies ---
ts_idx = rng.choice(df_raw.index, size=15, replace=False)
for i in ts_idx:
    df_raw.loc[i, 'ProductionDate'] = pd.Timestamp('2026-01-01') + pd.Timedelta(days=int(rng.integers(200, 400)))
    issue_log.append({'BatchID': df_raw.loc[i, 'BatchID'], 'IssueType': 'Timestamp Anomaly', 'Severity': 'Medium',
                       'Detail': 'ProductionDate falls outside expected production window'})

# --- Issue 5: Orphan records (missing station link) ---
orph_idx = rng.choice(df_raw.index, size=20, replace=False)
for i in orph_idx:
    df_raw.loc[i, 'StationID'] = None
    issue_log.append({'BatchID': df_raw.loc[i, 'BatchID'], 'IssueType': 'Orphan Record', 'Severity': 'High',
                       'Detail': 'StationID missing - batch cannot be traced to a production station'})

issue_log_df = pd.DataFrame(issue_log)
print(f"Injected {len(issue_log_df)} issues across {issue_log_df['BatchID'].nunique()} batches")
print(f"Raw table now has {df_raw.shape[0]} rows (was {df_clean.shape[0]}) due to injected duplicates")


Injected 150 issues across 147 batches
Raw table now has 3265 rows (was 3240) due to injected duplicates


## Step 4 — Build the automated Data Integrity Checker

This is the core "data integrity" engine. In a real system, this would run automatically every time new data lands (on ingestion), flagging problems before they reach reports or decisions. We re-detect the same issue categories purely by scanning the data — not by looking at our injected list — the way a real monitor would.


In [9]:
def run_integrity_checks(data):
    """Scans a batch dataset and returns every detected integrity issue."""
    findings = []

    # Check 1: duplicate BatchIDs
    dupe_ids = data['BatchID'][data['BatchID'].duplicated(keep=False)].unique()
    for bid in dupe_ids:
        findings.append({'BatchID': bid, 'IssueType': 'Duplicate Record', 'Severity': 'High'})

    # Check 2: missing values in critical fields
    critical_fields = ['SupplierQuality', 'DeliveryDelay', 'MaintenanceHours', 'StationID']
    for field in critical_fields:
        missing = data[data[field].isna()]
        for bid in missing['BatchID']:
            issue = 'Orphan Record' if field == 'StationID' else 'Missing Value'
            findings.append({'BatchID': bid, 'IssueType': issue,
                              'Severity': 'High' if field == 'StationID' else 'Medium'})

    # Check 3: out-of-range values
    oor = data[(data['DeliveryDelay'] < 0) | (data['DefectRate'] > 100) | (data['SupplierQuality'] > 100)]
    for bid in oor['BatchID']:
        findings.append({'BatchID': bid, 'IssueType': 'Out-of-Range Value', 'Severity': 'High'})

    # Check 4: timestamp anomalies (outside expected production window)
    ts_bad = data[(data['ProductionDate'] < '2026-01-01') | (data['ProductionDate'] > '2026-07-01')]
    for bid in ts_bad['BatchID']:
        findings.append({'BatchID': bid, 'IssueType': 'Timestamp Anomaly', 'Severity': 'Medium'})

    return pd.DataFrame(findings)

detected = run_integrity_checks(df_raw)
print("Issues detected by the automated checker:")
detected.groupby('IssueType').size().sort_values(ascending=False)


Issues detected by the automated checker:


,0
IssueType,
Missing Value,60
Out-of-Range Value,30
Duplicate Record,25
Orphan Record,20
Timestamp Anomaly,16


## Step 5 — Score each batch's data integrity health

Not every issue is equally serious — a missing optional field is less severe than a batch that can't be traced to a station at all. We weight issues by severity and compute a 0–100 **Integrity Score** per batch, similar to a credit score: 100 = perfectly clean, lower = more/worse issues.


In [10]:
severity_weight = {'High': 15, 'Medium': 7, 'Low': 3}
detected['Weight'] = detected['Severity'].map(severity_weight)

batch_penalty = detected.groupby('BatchID')['Weight'].sum().reset_index(name='PenaltyPoints')
batch_penalty['IntegrityScore'] = (100 - batch_penalty['PenaltyPoints']).clip(lower=0)

print(f"Average integrity score across flagged batches: {batch_penalty['IntegrityScore'].mean():.1f}")
batch_penalty.sort_values('IntegrityScore').head()


Average integrity score across flagged batches: 88.8


,BatchID,PenaltyPoints,IntegrityScore
131,BATCH-002766,45,55
27,BATCH-000592,30,70
4,BATCH-000071,15,85
0,BATCH-000022,15,85
12,BATCH-000277,15,85


## Step 6 — Predict defect risk with a Random Forest model

Now the predictive piece: using the **clean** dataset (`df_clean`), train a classifier that predicts `DefectStatus` (did this batch have a defect?) from the process and quality metrics. We use Random Forest because it handles mixed-scale numeric features well and gives us feature importance for free — useful for explaining *why* a batch is flagged as risky, not just that it is.


In [11]:
feature_cols = ['ProductionVolume', 'ProductionCost', 'SupplierQuality', 'DeliveryDelay',
                 'DefectRate', 'QualityScore', 'MaintenanceHours', 'DowntimePercentage',
                 'InventoryTurnover', 'StockoutRate', 'WorkerProductivity', 'SafetyIncidents',
                 'EnergyConsumption', 'EnergyEfficiency', 'AdditiveProcessTime', 'AdditiveMaterialCost']

X = df_clean[feature_cols]
y = df_clean['DefectStatus']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy: ", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:   ", round(recall_score(y_test, y_pred), 4))
print("F1 score: ", round(f1_score(y_test, y_pred), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy:  0.9506
Precision: 0.9508
Recall:    0.9927
F1 score:  0.9713

Confusion matrix:
[[ 75  28]
 [  4 541]]


**Reading the results:** ~95% accuracy matches published benchmarks on this exact dataset (Random Forest is consistently the strongest model here — see IRMAK 2025). Precision and recall both being high means the model isn't just guessing the majority class; it's genuinely separating defective from non-defective batches.

Now we apply the trained model to **every** batch (not just the test set) to get a risk score we can put on the dashboard.


In [12]:
df_clean['PredictedDefectProbability'] = model.predict_proba(X)[:, 1]
df_clean['PredictedDefectStatus'] = model.predict(X)
df_clean['RiskTier'] = pd.cut(df_clean['PredictedDefectProbability'],
                                bins=[-0.01, 0.5, 0.75, 1.0],
                                labels=['Low', 'Medium', 'High'])

feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

feature_importance.head(10)


,Feature,Importance
6,MaintenanceHours,0.273593
4,DefectRate,0.216339
5,QualityScore,0.148024
0,ProductionVolume,0.093134
15,AdditiveMaterialCost,0.031238
9,StockoutRate,0.030603
12,EnergyConsumption,0.026746
13,EnergyEfficiency,0.025421
2,SupplierQuality,0.024746
8,InventoryTurnover,0.024249


## Step 7 — Merge everything into one master table and export for Tableau

The final dataset joins the batch context, the ML risk scores, and the data integrity scores into a single table — this is what "traceability" looks like in practice: every batch carries its full history (who made it, when, its risk level, and whether its data can even be trusted) in one place.

We export four files, each built for a specific piece of the Tableau dashboard:

- `batches_master.csv` — one row per batch: full context + defect prediction + integrity score
- `integrity_issues_log.csv` — every individual issue detected, for a drill-down table
- `supplier_summary.csv` — aggregated risk/integrity by supplier, for supplier comparison charts
- `feature_importance.csv` — what drives defect risk, for an explainability chart


In [13]:
df_final = df_clean.merge(batch_penalty[['BatchID', 'IntegrityScore']], on='BatchID', how='left')
df_final['IntegrityScore'] = df_final['IntegrityScore'].fillna(100)  # untouched batches = perfect score

out_dir = 'output'
os.makedirs(out_dir, exist_ok=True)

df_final.to_csv(f'{out_dir}/batches_master.csv', index=False)
detected.drop_duplicates().to_csv(f'{out_dir}/integrity_issues_log.csv', index=False)
feature_importance.to_csv(f'{out_dir}/feature_importance.csv', index=False)

supplier_summary = df_final.groupby('SupplierName').agg(
    AvgDefectRate=('DefectRate', 'mean'),
    AvgIntegrityScore=('IntegrityScore', 'mean'),
    AvgPredictedRisk=('PredictedDefectProbability', 'mean'),
    BatchCount=('BatchID', 'count')
).reset_index().sort_values('AvgPredictedRisk', ascending=False)
supplier_summary.to_csv(f'{out_dir}/supplier_summary.csv', index=False)

print("Exported files:")
for f in os.listdir(out_dir):
    print(' -', f)


Exported files:
 - integrity_issues_log.csv
 - feature_importance.csv
 - batches_master.csv
 - supplier_summary.csv


## Step 8 — Quick sanity-check summary

A few numbers worth quoting in your resume bullet or interview:


In [14]:
print(f"Total batches analyzed: {len(df_final):,}")
print(f"Overall defect rate: {(df_final['DefectStatus'].mean()*100):.1f}%")
print(f"Model accuracy: {accuracy_score(y_test, y_pred)*100:.1f}%")
print(f"Batches flagged with data integrity issues: {detected['BatchID'].nunique()} ({detected['BatchID'].nunique()/len(df_final)*100:.1f}%)")
print(f"Average integrity score (flagged batches only): {batch_penalty['IntegrityScore'].mean():.1f} / 100")
print(f"\nTop defect-risk supplier: {supplier_summary.iloc[0]['SupplierName']} (avg predicted risk: {supplier_summary.iloc[0]['AvgPredictedRisk']:.2f})")


Total batches analyzed: 3,240
Overall defect rate: 84.0%
Model accuracy: 95.1%
Batches flagged with data integrity issues: 148 (4.6%)
Average integrity score (flagged batches only): 88.8 / 100

Top defect-risk supplier: Orion Parts (avg predicted risk: 0.83)


## Next step: Tableau dashboard

Open Tableau Public / Tableau Desktop and connect to `batches_master.csv`, `integrity_issues_log.csv`, and `supplier_summary.csv`. See the separate **Tableau Build Guide** for exact step-by-step chart instructions.
